In [ ]:
import os

# Set KaggleHub cache to a directory inside /content/
os.environ["KAGGLEHUB_CACHE"] = "./content/data"

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
import glob
import os
from matplotlib import pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import DataLoader


In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask

In [ ]:
# TO DO
from sklearn.model_selection import train_test_split

image_paths = glob.glob(f"{os.path.join(path, 'dataset')}/images/*.jpg")
masks = glob.glob(f"{os.path.join(path, 'dataset')}/masks/*.png")

train_image_paths, test_image_paths, train_masks, test_masks = train_test_split(image_paths, masks, test_size=0.2, shuffle=True)


# Custom Dataset Class
class SegmentationDataset(Dataset):
    def __init__(self, image_paths, masks, transform=None, target_transform=None):
        self.transform = transform
        self.target_transform = target_transform

        self.image_paths = image_paths
        self.masks = masks


    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.masks[idx]

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("P")  # Convert mask to grayscale (1 channel = binary segmentation) -> L otherwise make it into P for multiclass as i found that it works like that (had some weird error earlier)

        if self.transform:
            image = self.transform(image)

        if self.target_transform:
            mask = self.target_transform(mask)

        # Replace mask values with remapped values -> use the other function if it wasn't binary
        mask = remap_mask(mask)

        return image, mask       # In image classification datasets, we return image and label. Here, we return image and mask

# Define transforms for images and masks
image_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((256, 256)),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Standard ImageNet normalization
])

mask_transforms = transforms.Compose([
    transforms.Resize((256, 256), interpolation=transforms.InterpolationMode.NEAREST),  # Keep segmentation masks intact
    transforms.PILToTensor(),

])

train_dataset = SegmentationDataset(train_image_paths, train_masks, transform= image_transforms, target_transform= mask_transforms)
test_dataset = SegmentationDataset(test_image_paths, test_masks, transform= image_transforms, target_transform= mask_transforms)

train_loader = DataLoader(train_dataset, 4, shuffle= True, num_workers= 0)
test_loader = DataLoader(test_dataset, 4, shuffle= False, num_workers= 0)

print(f"train {len(train_dataset)}, test {len(test_dataset)}")



# Display some images with their masks
for i in range(3):
    img, mask = train_dataset[i]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    axes[0].imshow(img)
    axes[0].set_title("Image")
    axes[0].axis("off")
    axes[1].imshow(mask.permute(1,2,0), cmap="gray")
    axes[1].set_title("Segmentation Mask")
    axes[1].axis("off")
    plt.show()



In [ ]:
# TO DO
import segmentation_models_pytorch as smp

# Define U-Net Model
device = "cuda" if torch.cuda.is_available() else 'cpu'
print(device)
model = smp.Unet(
    encoder_name="efficientnet-b1",  # Pretrained encoder (backbone)
    encoder_weights="imagenet",  # Use ImageNet weights
    in_channels=3,  # RGB images
    classes=8,  # Binary segmentation (1 output channel) ***change it in case of multiclass seg***
).to(device)

In [ ]:
# TO DO
from tqdm import tqdm


def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    total_loss = 0

    for images, masks in tqdm(dataloader):
        images, masks = images.to(device), masks.to(device).to(torch.long)

        outputs = model(images)

        # print(outputs.dtype)
        # print(masks.dtype)#
        # print(outputs.shape)
        # print(masks.shape)
        loss = criterion(outputs, masks.squeeze(1)) # in binary do not squeeze

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for images, masks in dataloader:
            images, masks = images.to(device), masks.to(device).to(torch.long) # change it to float in binary seg

            outputs = model(images)
            loss = criterion(outputs, masks.squeeze(1)) # squeeze the mask dim=1 (channels) in multiclass (found out while studying) as crossentropy wants the mask like this to be able to calc loss while having the ouput 8 channels (this is to reduce vram consumption i think)
            # in binary do not squeeze

            total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
# TO DO
from torch import nn
from torch.optim import AdamW

criterion = nn.CrossEntropyLoss() # change based on the task (binary classification or multiclass)
optimizer = AdamW(model.parameters(), lr=0.0001)

num_epochs = 4
train_losses = []
val_losses = []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs}: Train Loss = {train_loss:.4f}, Val Loss = {val_loss:.4f}")

import matplotlib.pyplot as plt

plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.show()


In [ ]:
# TO DO
# TO DO
import random
import matplotlib.pyplot as plt
import numpy as np

# Function to denormalize images
def denormalize(img):
    mean = np.array([0.485, 0.456, 0.406])  # ImageNet mean
    std = np.array([0.229, 0.224, 0.225])  # ImageNet std
    img = img.numpy().transpose(1, 2, 0)  # Convert to HWC
    img = img * std + mean  # Reverse normalization
    img = np.clip(img, 0, 1)  # Clip values to [0,1]
    return img

# Set model to evaluation mode
model.eval()

# Get some test samples
test_samples = random.sample(range(len(test_dataset)), 5)

for idx in test_samples:
    img, mask = test_dataset[idx]

    with torch.no_grad():
        pred_mask = model(img.unsqueeze(0).to(device))  # Forward pass unsqueeze to add batchsize of 1

    # pred_mask = (pred_mask >= 0.5).cpu().squeeze().numpy() # this is in case of binary classification
    # in case of multi class we use argmax use it in the channels only
    #torch.argmax(pred_mask, dim=1).cpu().squeeze().numpy() #another way using torch fun
    pred_mask = pred_mask.argmax(dim=1).cpu().squeeze().numpy()

    # Display images
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # Original Image (Denormalized)
    axes[0].imshow(denormalize(img))
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    # Ground Truth Mask
    axes[1].imshow(mask.squeeze(), cmap="gray")
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis("off")

    # print(type(pred_mask))
    # Predicted Mask
    axes[2].imshow(pred_mask, cmap="gray")
    axes[2].set_title("Predicted Mask")
    axes[2].axis("off")

    plt.show()
